In [1]:
!pip show tensorflow

import tensorflow as tf

print("TensorFlow sürümü:", tf.__version__)
print("GPU sayısı:", len(tf.config.list_physical_devices('GPU')))
print("GPU cihazları:", tf.config.list_physical_devices('GPU'))

Name: tensorflow
Version: 2.20.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: absl-py, astunparse, flatbuffers, gast, google_pasta, grpcio, h5py, keras, libclang, ml_dtypes, numpy, opt_einsum, packaging, protobuf, requests, setuptools, six, tensorboard, termcolor, typing_extensions, wrapt
Required-by: dopamine_rl, tensorflow-text, tf_keras, ydf_tf
TensorFlow sürümü: 2.20.0
GPU sayısı: 1
GPU cihazları: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import zipfile

zip_dosyasi = "/content/iskelet_goruntuleri_filtreli_128.zip"

hedef_klasor = "veriseti"

with zipfile.ZipFile(zip_dosyasi, 'r') as zip_ref:
    zip_ref.extractall(hedef_klasor)
    print(".")


.


In [3]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

DATASET_DIR = "/content/veriseti/iskelet_goruntuleri_filtreli_128"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 50

train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Sınıflar ({num_classes} adet): {class_names}")

with open("labels.txt", "w", encoding="utf-8") as f:
    for label in class_names:
        f.write(f"{label}\n")


AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

model = models.Sequential([
    layers.RandomRotation(0.1, input_shape=(128, 128, 3), name="random_rotation"),
    layers.RandomTranslation(0.1, 0.1, name="random_translation"),
    layers.RandomZoom(0.1, name="random_zoom"),

    layers.Rescaling(1./255, name="rescaling"),

    layers.Conv2D(32, (3, 3), padding='valid', activation='relu', name="conv2d"),
    layers.MaxPooling2D((2, 2), name="max_pooling2d"),

    layers.Conv2D(64, (3, 3), padding='valid', activation='relu', name="conv2d_1"),
    layers.MaxPooling2D((2, 2), name="max_pooling2d_1"),

    layers.Conv2D(128, (3, 3), padding='valid', activation='relu', name="conv2d_2"),
    layers.MaxPooling2D((2, 2), name="max_pooling2d_2"),

    layers.Flatten(name="flatten"),
    layers.Dense(128, activation='relu', name="dense"),
    layers.Dropout(0.5, name="dropout"),
    layers.Dense(num_classes, activation='softmax', name="dense_1")
])

model.summary()

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks = [
    ModelCheckpoint("harfModeli.keras", save_best_only=True, monitor='val_accuracy', mode='max'),
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("tamam")

Found 48796 files belonging to 26 classes.
Using 39037 files for training.
Found 48796 files belonging to 26 classes.
Using 9759 files for validation.
Sınıflar (26 adet): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'Y', 'Z', 'del', 'nothing', 'space']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_rotation                 │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation              │ (None, 128, 128, 3)    │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 26)             │         3,354 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,307,994 (12.62 MB)

 Trainable params: 3,307,994 (12.62 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 41s 28ms/step - accuracy: 0.4225 - loss: 1.7764 - val_accuracy: 0.7764 - val_loss: 0.7525
Epoch 2/50
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 25s 20ms/step - accuracy: 0.5866 - loss: 1.2552 - val_accuracy: 0.8315 - val_loss: 0.5501
Epoch 3/50
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - accuracy: 0.6524 - loss: 1.0694 - val_accuracy: 0.8609 - val_loss: 0.4641
Epoch 4/50
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - accuracy: 0.6884 - loss: 0.9636 - val_accuracy: 0.8794 - val_loss: 0.4130
Epoch 5/50
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - accuracy: 0.7133 - loss: 0.8866 - val_accuracy: 0.8977 - val_loss: 0.3581
Epoch 6/50
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - accuracy: 0.7352 - loss: 0.8234 - val_accuracy: 0.9074 - val_loss: 0.3330
Epoch 7/50
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - accuracy: 0.7527 - loss: 0.7735 - val_accuracy: 0.9195 - val_loss: 0.3039
Epoch 8/50
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - accuracy: 0.7606 -